# Start

# Colab Step 1 — Check GPU

In [1]:
!nvidia-smi

Sun Sep  6 14:14:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# Colab Step 2 — Clone PVT-CASCADE

In [3]:
!git clone https://github.com/SLDGroup/CASCADE.git
%cd CASCADE

Cloning into 'CASCADE'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 136 (delta 64), reused 35 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.05 MiB | 3.45 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/CASCADE


In [4]:
!pip install -q timm
!pip install -q opencv-python
!pip install -q scikit-image
!pip install -q networkx
!pip install -q python-louvain

# Colab Step 3 — Download the pretrained PVTv2-B2 weights

In [5]:
!mkdir -p pretrained_pth/pvt

In [6]:
!wget -O pretrained_pth/pvt/pvt_v2_b2.pth \
https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth

--2026-09-06 14:15:17--  https://github.com/whai362/PVT/releases/download/v2/pvt_v2_b2.pth
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/341748701/0adeb500-d9a9-11eb-9cec-afdf37fdf2ec?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-09-06T14%3A53%3A44Z&rscd=attachment%3B+filename%3Dpvt_v2_b2.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-09-06T13%3A53%3A06Z&ske=2026-09-06T14%3A53%3A44Z&sks=b&skv=2018-11-09&sig=dNjCjWLE44WaXBrHawMpxGE7ptaQi%2Fm3JZ1gwVPdAfU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4ODcwNTkxNywibmJmIjoxNzg4NzA0MTE3LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2lu

In [7]:
!ls -lh pretrained_pth/pvt/

total 97M
-rw-r--r-- 1 root root 97M Dec  7  2021 pvt_v2_b2.pth


In [8]:
!pip install -q ml-collections

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.6 MB/s eta 0:00:00


# Colab Step 4 — Check the PVT-CASCADE model

In [9]:
import torch
from lib.networks import PVT_CASCADE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PVT_CASCADE(n_class=1)
model = model.to(device)

print("Model loaded successfully.")

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/content/CASCADE/lib/pvtv2.py:387: UserWarning: Overwriting pvt_v2_b0 in registry with lib.pvtv2.pvt_v2_b0. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  @register_model
/content/CASCADE/lib/pvtv2.py:397: UserWarning: Overwriting pvt_v2_b1 in registry with lib.pvtv2.pvt_v2_b1. This is because the name being registered conflicts with an existing name. Please check if this is not expect

Model loaded successfully.


# Colab Step 5 — Test one fake image

In [10]:
x = torch.randn(1, 3, 352, 352).to(device)

with torch.no_grad():
    outputs = model(x)

print(type(outputs))
print(len(outputs))

for i, output in enumerate(outputs):
    print(f"Output {i+1}: {output.shape}")

<class 'tuple'>
4
Output 1: torch.Size([1, 1, 352, 352])
Output 2: torch.Size([1, 1, 352, 352])
Output 3: torch.Size([1, 1, 352, 352])
Output 4: torch.Size([1, 1, 352, 352])


In [11]:
# Install kagglehub if needed
!pip install -q kagglehub

import kagglehub

# Download Kvasir-SEG from Kaggle
dataset_path = kagglehub.dataset_download("fkarimovv/kvasir-seg")

print("Dataset downloaded to:")
print(dataset_path)

100%|██████████| 44.0M/44.0M [00:00<00:00, 81.8MB/s]

Extracting files...


Dataset downloaded to:
/root/.cache/kagglehub/datasets/fkarimovv/kvasir-seg/versions/1


In [13]:
import os

for root, dirs, files in os.walk(dataset_path):
    image_count = sum(
        f.lower().endswith((".jpg", ".jpeg", ".png"))
        for f in files
    )

    if image_count > 0:
        print(root, "->", image_count, "images")

/root/.cache/kagglehub/datasets/fkarimovv/kvasir-seg/versions/1/Kvasir-SEG/masks -> 1000 images
/root/.cache/kagglehub/datasets/fkarimovv/kvasir-seg/versions/1/Kvasir-SEG/images -> 1000 images


# Colab Step 7 — Dataset class

In [14]:
import cv2
import numpy as np
import torch

from torch.utils.data import Dataset

In [15]:
class KvasirDataset(Dataset):

    def __init__(
        self,
        image_dir,
        mask_dir,
        image_names,
        image_size=352
    ):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_names = image_names
        self.image_size = image_size

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        # -------------------------
        # Image path
        # -------------------------

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        # -------------------------
        # Mask has SAME filename
        # -------------------------

        mask_path = os.path.join(
            self.mask_dir,
            image_name
        )

        # -------------------------
        # Read image
        # -------------------------

        image = cv2.imread(image_path)

        if image is None:
            raise RuntimeError(
                f"Could not read image: {image_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        # -------------------------
        # Read mask
        # -------------------------

        mask = cv2.imread(
            mask_path,
            cv2.IMREAD_GRAYSCALE
        )

        if mask is None:
            raise RuntimeError(
                f"Could not read mask: {mask_path}"
            )

        # -------------------------
        # Resize
        # -------------------------

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR
        )

        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST
        )

        # -------------------------
        # Normalize image
        # -------------------------

        image = image.astype(
            np.float32
        ) / 255.0

        # -------------------------
        # Binary mask
        # -------------------------

        mask = (
            mask > 127
        ).astype(np.float32)

        # -------------------------
        # HWC -> CHW
        # -------------------------

        image = torch.from_numpy(
            image.transpose(2, 0, 1)
        ).float()

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0).float()

        return image, mask

# Colab Step 8 — Split the dataset

In [16]:
# ============================================
# Colab Step 8 — Kvasir-SEG paths + split
# ============================================

import os
import numpy as np
import cv2
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split


# ------------------------------------------------
# 1. Set Kvasir-SEG directories
# ------------------------------------------------

image_dir = os.path.join(dataset_path, "Kvasir-SEG", "images")
mask_dir = os.path.join(dataset_path, "Kvasir-SEG", "masks")

print("Image directory:", image_dir)
print("Mask directory :", mask_dir)


# ------------------------------------------------
# 2. Check directories exist
# ------------------------------------------------

assert os.path.isdir(image_dir), f"Image directory not found: {image_dir}"
assert os.path.isdir(mask_dir), f"Mask directory not found: {mask_dir}"

print("\nDirectories found successfully.")


# ------------------------------------------------
# 3. Get all image filenames
# ------------------------------------------------

all_images = sorted([
    f for f in os.listdir(image_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
])

print("Total images:", len(all_images))


# ------------------------------------------------
# 4. Train / Validation / Test split
# ------------------------------------------------
# 70% train
# 15% validation
# 15% test

train_images, temp_images = train_test_split(
    all_images,
    test_size=0.30,
    random_state=42
)

val_images, test_images = train_test_split(
    temp_images,
    test_size=0.50,
    random_state=42
)


print("\nDataset split:")
print("Train      :", len(train_images))
print("Validation :", len(val_images))
print("Test       :", len(test_images))


# ------------------------------------------------
# 5. Verify no overlap
# ------------------------------------------------

train_set = set(train_images)
val_set = set(val_images)
test_set = set(test_images)

print("\nOverlap checks:")
print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test      :", len(train_set & test_set))
print("Validation ∩ Test :", len(val_set & test_set))

# ============================================
# Create Dataset objects
# ============================================

train_dataset = KvasirDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    image_names=train_images,
    image_size=352
)

val_dataset = KvasirDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    image_names=val_images,
    image_size=352
)

test_dataset = KvasirDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    image_names=test_images,
    image_size=352
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Image directory: /root/.cache/kagglehub/datasets/fkarimovv/kvasir-seg/versions/1/Kvasir-SEG/images
Mask directory : /root/.cache/kagglehub/datasets/fkarimovv/kvasir-seg/versions/1/Kvasir-SEG/masks

Directories found successfully.
Total images: 1000

Dataset split:
Train      : 700
Validation : 150
Test       : 150

Overlap checks:
Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0
Train: 700
Validation: 150
Test: 150


# Colab Step 9 — DataLoaders

In [17]:
from torch.utils.data import DataLoader

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

# Colab Step 10 — PVT-CASCADE loss

In [18]:
import torch.nn as nn
import torch.nn.functional as F

In [19]:
class StructureLoss(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, pred, mask):

        weit = 1 + 5 * torch.abs(
            F.avg_pool2d(
                mask,
                kernel_size=31,
                stride=1,
                padding=15
            ) - mask
        )

        wbce = F.binary_cross_entropy_with_logits(
            pred,
            mask,
            reduction="none"
        )

        wbce = (
            (weit * wbce).sum(dim=(2, 3))
            / weit.sum(dim=(2, 3))
        )

        pred = torch.sigmoid(pred)

        inter = (
            (pred * mask) * weit
        ).sum(dim=(2, 3))

        union = (
            (pred + mask) * weit
        ).sum(dim=(2, 3))

        wiou = 1 - (
            (inter + 1)
            / (union - inter + 1)
        )

        return (wbce + wiou).mean()

# Colab Step 11 — Training function

In [20]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    total_loss = 0.0

    for images, masks in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        p1, p2, p3, p4 = model(images)

        loss1 = criterion(p1, masks)
        loss2 = criterion(p2, masks)
        loss3 = criterion(p3, masks)
        loss4 = criterion(p4, masks)

        loss = (
            loss1 +
            loss2 +
            loss3 +
            loss4
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# Colab Step 12 — Metrics

In [21]:
@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    total_dice = 0.0
    total_iou = 0.0

    n_images = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        p1, p2, p3, p4 = model(images)

        logits = p1 + p2 + p3 + p4

        probs = torch.sigmoid(logits)

        preds = (probs > 0.5).float()

        intersection = (
            preds * masks
        ).sum(dim=(1, 2, 3))

        pred_area = preds.sum(
            dim=(1, 2, 3)
        )

        gt_area = masks.sum(
            dim=(1, 2, 3)
        )

        dice = (
            2 * intersection + 1e-7
        ) / (
            pred_area +
            gt_area +
            1e-7
        )

        union = (
            pred_area +
            gt_area -
            intersection
        )

        iou = (
            intersection + 1e-7
        ) / (
            union + 1e-7
        )

        total_dice += dice.sum().item()
        total_iou += iou.sum().item()

        n_images += images.size(0)

    return (
        total_dice / n_images,
        total_iou / n_images
    )

# Colab Step 13 — Create model

In [22]:
model = PVT_CASCADE(n_class=1).to(device)

criterion = StructureLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Colab Step 14 — Train

In [26]:
num_epochs = 5

best_dice = 0.0

for epoch in range(num_epochs):

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_dice, val_iou = evaluate(
        model,
        val_loader,
        device
    )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {train_loss:.4f} "
        f"Val Dice: {val_dice:.4f} "
        f"Val IoU: {val_iou:.4f}"
    )

    if val_dice > best_dice:

        best_dice = val_dice

        torch.save(
            model.state_dict(),
            "best_pvt_cascade_kvasir.pth"
        )

        print("Saved best model.")

Epoch [1/5] Loss: 1.6134 Val Dice: 0.9091 Val IoU: 0.8507
Saved best model.
Epoch [2/5] Loss: 1.5745 Val Dice: 0.8809 Val IoU: 0.8113
Epoch [3/5] Loss: 1.4324 Val Dice: 0.8994 Val IoU: 0.8362
Epoch [4/5] Loss: 1.2931 Val Dice: 0.9056 Val IoU: 0.8472
Epoch [5/5] Loss: 1.1544 Val Dice: 0.9005 Val IoU: 0.8392


# Colab Step 15 — Test the best model

In [27]:
model.load_state_dict(
    torch.load(
        "best_pvt_cascade_kvasir.pth",
        map_location=device
    )
)

test_dice, test_iou = evaluate(
    model,
    test_loader,
    device
)

print("Test Dice:", test_dice)
print("Test IoU :", test_iou)

Test Dice: 0.8958991209665934
Test IoU : 0.838045883178711


# END